# Catégorie Science

## Étape 1 — le périmètre
•	le nombre de livres annoncé en haut de la page : 14
•	le nombre de pages : 1 page (absence de Page 1 of N)

## Étape 2 — le bloc qui se répète
document.querySelectorAll('article.product_pod').length

>14

## Étape 3 — le tableau des cinq données

| Donnée             | Sélecteur      | Contenu ou attribut ?   |
|:-------------------|:---------------|:------------------------|
| titre              | `h3 a`         | Attribut `title`        |
| prix               | `p.price_color`| Contenu                 |
| disponibilité      | `p.instock`    | Contenu                 |
| note               | `p.star-rating`| Attribut `class`        |
| lien vers la fiche | `h3 a`         | Attribut `href`         |

## Étape 4 — deux sélecteurs testés
document.querySelectorAll('article.product_pod p.price_color').length

> 14

## Étape 5 — la page suivante
Une seule page pour la catégorie Science. Lors du script nous devrons « gérer » cette erreur  un script qui cherche ce lien et qui ne le trouve pas devra renvoyer la mention « 1 unique page » (et non pas une erreur)

## Étape 6 — le contrôle du code source
Selection du prix 42.96 livre (book n°1 : “The most perfect thing Inside : (and Outside) a Bird's Egg”). 
Prix trouvé dans le code source avec CTRL + F  lorsque l’on fera appel au request sur le code source pour récupérer le prix, il sera possible de récupérer cette information.

## Étape 7 — deux données de la fiche détaillée
Information présente sur cette page et absente de la liste : 
-	Availability : le nombre de produit en stock 
-	Number of reviews
Ces deux informations se trouvent dans le tableau Product Information.
Au niveau de l’inspecteur elles se trouvent au niveau du selecteur « table tbody tr th td »


In [78]:
# Étape 2 — une page, pour valider la chaîne
import requests
URL = "https://books.toscrape.com/catalogue/category/books/science_22/index.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (kelly.cheniara@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status : ", r.status_code)
print("longueur : ", len(r.text), "caractère")
print("debut : ", r.text[:60])


status :  200
longueur :  42533 caractère
debut :  

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [79]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))
                     


fiches trouvees : 14


In [80]:
fiche = fiches[0]

prix = fiche.select_one("p.price_color").get_text(strip=True)
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix : ", prix)
print("titre : ", titre)
print("href : ", href)

prix :  £42.96
titre :  The Most Perfect Thing: Inside (and Outside) a Bird's Egg
href :  ../../../the-most-perfect-thing-inside-and-outside-a-birds-egg_938/index.html


In [81]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes : ", classes)

NOTES = {"One" : 1, "Two" : 2,"Three": 3, "Four": 4, "Five": 5}
note = NOTES[classes[1]]
print("note : ", note)

classes :  ['star-rating', 'Four']
note :  4


In [82]:
from urllib.parse import urljoin
print(urljoin(URL, href))

https://books.toscrape.com/catalogue/the-most-perfect-thing-inside-and-outside-a-birds-egg_938/index.html


In [83]:
def lire_fiche(fiche, url_page): 
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre" : lien["title"],
        "prix" : fiche.select_one("p.price_color").get_text(strip=True),
        "dispo" : fiche.select_one("p.instock").get_text(strip=True),
        "note" : NOTES[classes[1]],
        "url" : urljoin(url_page, lien["href"])
    }


In [84]:
#Étape 3 — toutes les pages de liste
import time
def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres=[]
    page=0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f" page {page} : status {r.status_code}, on s'arrete")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f,url) for f in fiches]
        print(f" page {page} : {len(fiches)} livres (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres

livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/category/books/science_22/index.html")
print ("collecte : ", len(livres), "livres")


 page 1 : 14 livres (total 14)
collecte :  14 livres


In [85]:
#etape 4

def lire_detail(url) : 
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200 : 
        return {"categorie" : None, "stock" : None}

    soup = BeautifulSoup(r.text, "html.parser")

    infos={}
    for ligne in soup.select('table.table.table-striped tr') : 
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val : 
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    time.sleep(1)

    return {
        "categorie" : categorie,
        "stock" : infos.get("Availability"),
        "upc" : infos.get("UPC"),
        }


    

In [86]:
liste_url = []
for livre in livres : 
    liste_url.append(livre["url"])
print(liste_url)

['https://books.toscrape.com/catalogue/the-most-perfect-thing-inside-and-outside-a-birds-egg_938/index.html', 'https://books.toscrape.com/catalogue/immunity-how-elie-metchnikoff-changed-the-course-of-modern-medicine_900/index.html', 'https://books.toscrape.com/catalogue/sorting-the-beef-from-the-bull-the-science-of-food-fraud-forensics_736/index.html', 'https://books.toscrape.com/catalogue/tipping-point-for-planet-earth-how-close-are-we-to-the-edge_643/index.html', 'https://books.toscrape.com/catalogue/the-fabric-of-the-cosmos-space-time-and-the-texture-of-reality_572/index.html', 'https://books.toscrape.com/catalogue/diary-of-a-citizen-scientist-chasing-tiger-beetles-and-other-new-ways-of-engaging-the-world_517/index.html', 'https://books.toscrape.com/catalogue/the-origin-of-species_499/index.html', 'https://books.toscrape.com/catalogue/the-grand-design_405/index.html', 'https://books.toscrape.com/catalogue/peak-secrets-from-the-new-science-of-expertise_389/index.html', 'https://books

In [87]:
import re
def nombre_en_stock(texte) : 
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0

In [88]:
print(livres)

[{'titre': "The Most Perfect Thing: Inside (and Outside) a Bird's Egg", 'prix': '£42.96', 'dispo': 'In stock', 'note': 4, 'url': 'https://books.toscrape.com/catalogue/the-most-perfect-thing-inside-and-outside-a-birds-egg_938/index.html'}, {'titre': 'Immunity: How Elie Metchnikoff Changed the Course of Modern Medicine', 'prix': '£57.36', 'dispo': 'In stock', 'note': 5, 'url': 'https://books.toscrape.com/catalogue/immunity-how-elie-metchnikoff-changed-the-course-of-modern-medicine_900/index.html'}, {'titre': 'Sorting the Beef from the Bull: The Science of Food Fraud Forensics', 'prix': '£44.74', 'dispo': 'In stock', 'note': 4, 'url': 'https://books.toscrape.com/catalogue/sorting-the-beef-from-the-bull-the-science-of-food-fraud-forensics_736/index.html'}, {'titre': 'Tipping Point for Planet Earth: How Close Are We to the Edge?', 'prix': '£37.55', 'dispo': 'In stock', 'note': 1, 'url': 'https://books.toscrape.com/catalogue/tipping-point-for-planet-earth-how-close-are-we-to-the-edge_643/ind

In [89]:
# Étape 5 — assembler et nettoyer
import pandas as pd
assembler=[]
for livre in livres : 
        detail_livre = lire_detail(livre["url"])
        livre_info = {
                "titre" : livre["titre"],
                "categorie" : detail_livre["categorie"],
                "prix_gbp" : livre["prix"],
                "note" : livre["note"],
                "stock" : nombre_en_stock(detail_livre["stock"]),
                "upc" : detail_livre["upc"],
                "url" : livre["url"]
        }
        assembler.append(livre_info)


df = pd.DataFrame(assembler)
print(df)

                                                titre categorie prix_gbp  \
0   The Most Perfect Thing: Inside (and Outside) a...   Science   £42.96   
1   Immunity: How Elie Metchnikoff Changed the Cou...   Science   £57.36   
2   Sorting the Beef from the Bull: The Science of...   Science   £44.74   
3   Tipping Point for Planet Earth: How Close Are ...   Science   £37.55   
4   The Fabric of the Cosmos: Space, Time, and the...   Science   £55.91   
5   Diary of a Citizen Scientist: Chasing Tiger Be...   Science   £28.41   
6                               The Origin of Species   Science   £10.01   
7                                    The Grand Design   Science   £13.76   
8     Peak: Secrets from the New Science of Expertise   Science   £16.28   
9   The Elegant Universe: Superstrings, Hidden Dim...   Science   £13.03   
10  The Disappearing Spoon: And Other True Tales o...   Science   £57.35   
11  Surely You're Joking, Mr. Feynman!: Adventures...   Science   £25.83   
12          

In [90]:
df["prix"] = df["prix_gbp"].str.replace("£", "", regex=False).astype(float)

In [91]:
# Étape 6 — analyser

#Combien de livre et quel stock total
print(f"Le nombre de ligne sur la catégorie Science est {len(df)}.")
print(f"Le stock total sur la catégorie Science est de {df["stock"].sum()} livres.")
print(f"Le prix moyen sur la catégorie Science est de {round(df["prix"].mean(),2)}.")
print(f"Le prix moyen sur la catégorie Science est de {df["prix"].max()}.")
print(f"Les notes se répartissent selon le tableau suivant : ")
df_groupe = df.groupby("note")["note"].count()
print(df_groupe)

df["categorie"].value_counts()

#Nous avons bien 14 valeurs dans la section Science. 
# #Si nous avions d'autres catégories = selecteur pas positionné au bon niveau

Le nombre de ligne sur la catégorie Science est 14.
Le stock total sur la catégorie Science est de 105 livres.
Le prix moyen sur la catégorie Science est de 33.09.
Le prix moyen sur la catégorie Science est de 57.36.
Les notes se répartissent selon le tableau suivant : 
note
1    4
2    2
3    1
4    5
5    2
Name: note, dtype: int64


categorie
Science    14
Name: count, dtype: int64

In [92]:
#Etape 7
df.to_csv("../data/livres_science.csv", index=False, encoding="utf-8")
print()
print("livres_science.csv écrit --", len(df), "lignes")


livres_science.csv écrit -- 14 lignes
